In [34]:
from pyscf import gto, scf, cc

a = 2 # 2aB
nH = 2
atoms = ""
for i in range(nH):
    atoms += f"N {i*a:.5f} 0.00000 0.00000 \n"

mol = gto.M(atom=atoms, basis="sto6g", unit='B', spin=0, verbose=4)
mol.build()

mf = scf.RHF(mol)
mf.kernel()

mo = mf.stability()[0]
dm = mf.make_rdm1(mo,mf.mo_occ)
mf.kernel(dm0=dm)
mo = mf.stability()[0]
dm = mf.make_rdm1(mo,mf.mo_occ)
mf.kernel(dm0=dm)

nfrozen = 0
mycc = cc.CCSD(mf,frozen=nfrozen)
mycc.kernel()[0]

System: uname_result(system='Linux', node='yichi-thinkpad', release='4.4.0-26100-Microsoft', version='#7705-Microsoft Fri Jan 01 08:00:00 PST 2016', machine='x86_64')  Threads 12
Python 3.10.16 | packaged by conda-forge | (main, Dec  5 2024, 14:16:10) [GCC 13.3.0]
numpy 1.24.3  scipy 1.14.1  h5py 3.12.1
Date: Sun Feb 22 13:40:52 2026
PySCF version 2.12.1
PySCF path  /home/yichi/research/software/pyscf
GIT ORIG_HEAD a0665c4a7bf54e33f01295b3eea390be7a17d76d
GIT HEAD (branch master) f97393b29b0a541c155a68d55ee5b652ae7131d2

[ENV] PYSCF_EXT_PATH /home/yichi/research/software/pyscf
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 2
[INPUT] num. electrons = 14
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = B
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 N      0.000000000000   0.000000000000   0.00000000000

-0.1438354434456097

In [35]:
# example for PT2

options = {'n_eql': 3,
           'n_prop_steps': 50,
            'n_ene_blocks': 1,
            'n_sr_blocks': 5,
            'n_blocks': 50,
            'n_walkers': 1,
            'dt':0.005,
            'seed': 2,
            'walker_type': 'rhf',
            'trial': 'stoccsd2',
            'nslater': 100,
            'free_projection':False,
            'fp_abs': False,
            'group': False,
            'ad_mode': None,
            'use_gpu': False,
            }

from ad_afqmc.prop_unrestricted.mixed_wave import prep
import jax
jax.config.update("jax_enable_x64", True)
prep.prep_afqmc(mycc,chol_cut=1e-5)
# prop_unrestricted.run_afqmc(options,nproc=1)
option_file='options.bin'
import pickle
with open(option_file, 'wb') as f:
    pickle.dump(options, f)

#
# Preparing AFQMC calculation
# Calculating Cholesky integrals
# Finished calculating Cholesky integrals
#
# Size of the correlation space:
# Number of electrons: (7, 7)
# Number of basis functions: 10
# Number of Cholesky vectors: 42
#


In [36]:
import time
import numpy as np
from jax import random
from jax import numpy as jnp
from functools import partial 

ham_data, ham, prop, trial, wave_data, sampler, options = (prep._prep_afqmc())

init_time = time.time()

### initialize propagation
init_walkers = None
trial_rdm1 = trial.get_rdm1(wave_data)
if "rdm1" not in wave_data:
    wave_data["rdm1"] = trial_rdm1
ham_data = ham.build_measurement_intermediates(ham_data, trial, wave_data)
ham_data = ham.build_propagation_intermediates(ham_data, prop, trial, wave_data)

prop_data = prop.init_prop_data(trial, wave_data, ham_data, init_walkers)
if jnp.abs(jnp.sum(prop_data["overlaps"])) < 1.0e-6:
    raise ValueError(
        "Initial overlaps are zero. Pass walkers with non-zero overlap."
    )
prop_data["key"] = random.PRNGKey(options["seed"])

prop_data["overlaps"] = trial.calc_overlap(prop_data["walkers"], wave_data)
prop_data["n_killed_walkers"] = 0

e_init= jnp.real(trial.calc_energy(prop_data["walkers"], ham_data, wave_data)[0])
prop_data["e_estimate"] = e_init
prop_data["pop_control_ene_shift"] = prop_data["e_estimate"]

print(e_init)
print(e_init-mf.e_tot)

# norb: 10
# nelec: (7, 7)
# n_eql: 3
# n_prop_steps: 50
# n_ene_blocks: 1
# n_sr_blocks: 5
# n_blocks: 50
# n_walkers: 1
# dt: 0.005
# seed: 2
# walker_type: rhf
# trial: stoccsd2
# nslater: 100
# free_projection: False
# fp_abs: False
# group: False
# use_gpu: False
# n_exp_terms: 6
# n_batch: 1
# max_error: 0.001
-108.52346313844814
1.533424492095037e-05


In [29]:
# energy evaluator for <exp(T1)HF|(xtau + 1/2(xtau)^2)H|walker>
# exploitate the speed up by disconnected doubles

from jax import jit, lax, jvp
import opt_einsum as oe

# ad reference 
@jit
def _green(walker: jax.Array, slater: jax.Array) -> jax.Array:
    '''
    full green's function 
    <psi|a_p^dagger a_q|walker>/<exp(T1)HF|walker>
    '''
    green = (walker @ (
        jnp.linalg.inv(slater.T.conj() @ walker)
        ) @ slater.T.conj()).T
    return green

@jit
def _walker_olp(walker: jax.Array, slater: jax.Array):
    ''' 
    <psi|walker>
    '''
    olp = jnp.linalg.det(slater.T.conj() @ walker) ** 2
    return olp

@jit
def _ci_walker_olp(walker: jax.Array, slater: jax.Array, ci1: jax.Array, ci2: jax.Array) -> complex:
    ''' 
    <(1+ci1+ci2)psi|walker>
    = c_ia* <psi|ia|walker> + 1/2 c_iajb* <psi|ijab|walker>
    '''
    ci1 = ci1.conj()
    ci2 = ci2.conj()
    nocc = walker.shape[1]
    green_ov = _green(walker, slater)[:nocc, nocc:]
    o0 = _walker_olp(walker, slater)
    o1 = 2 * oe.contract("ia,ia-> ", ci1, green_ov, backend="jax")
    o2 = 2 * oe.contract("iajb,ia,jb->", ci2, green_ov, green_ov, backend="jax") \
        - oe.contract("iajb,ib,ja->", ci2, green_ov, green_ov, backend="jax")
    return (1.0 + o1 + o2) * o0

@jit
def _ci_exp_h1(x, h1_mod: jax.Array, walker: jax.Array, slater: jax.Array, ci1: jax.Array, ci2: jax.Array) -> complex:
    '''
    <exp(T1)HF|(1+ci1+ci2) exp(x*h1_mod)|walker>
    '''
    t = x * h1_mod
    walker_1x = walker + t.dot(walker)
    o_exp = _ci_walker_olp(walker_1x, slater, ci1, ci2)
    return o_exp 

@jit
def _ci_exp_h2(x, chol_i: jax.Array, walker: jax.Array, slater: jax.Array, ci1: jax.Array, ci2: jax.Array) -> complex:
    '''
    <exp(T1)HF|(1+ci1+ci2) exp(x*h2)|walker>
    '''
    walker_2x = (
            walker
            + x * chol_i.dot(walker)
            + x**2 / 2.0 * chol_i.dot(chol_i.dot(walker))
        )
    o_exp = _ci_walker_olp(walker_2x, slater, ci1, ci2)

    return o_exp

@jit
def _d2_ci_exp_h2i(chol_i: jax.Array, walker: jax.Array, slater: jax.Array, ci1: jax.Array, ci2: jax.Array):
    x = 0.0
    f = lambda a: _ci_exp_h2(a, chol_i, walker, slater, ci1, ci2)
    _, d2f = jax.jvp(lambda x: jax.jvp(f, [x], [1.0])[1], [x], [1.0])
    return d2f


@partial(jit, static_argnums=0)
def _calc_energy_ad(trial, walker, ham_data, wave_data, ci1, ci2):

    norb = trial.norb
    h0 = ham_data['h0']
    h1_mod = ham_data['h1_mod']
    chol = ham_data["chol"].reshape(-1, norb, norb)
    trial_slater = wave_data['mo_t']

    # o0 = _walker_olp(walker, trial_slater)

    # one body
    f1 = lambda a: _ci_exp_h1(a, h1_mod, walker, trial_slater, ci1, ci2)
    olp, d1_overlap = jvp(f1, [0.0], [1.0])

    # two body
    def scan_chol(carry, c):
        walker, trial_slater = carry
        return carry, _d2_ci_exp_h2i(c, walker, trial_slater, ci1, ci2)

    _, d2_overlap_i = lax.scan(scan_chol, (walker, trial_slater), chol)
    d2_overlap = jnp.sum(d2_overlap_i)/2

    # <psi|(1+ci1+ci2) (h1+h2)|walker> / <psi|1+ci1+ci2|walker>
    e12 = (d1_overlap + d2_overlap) / olp

    return olp, h0 + e12

In [52]:
@jit
def _ci_walker_olp_dc(walker: jax.Array, slater: jax.Array, ci1: jax.Array) -> complex:
    ''' 
    <(1+ci1+ci2)psi|walker> for disconnected doubles
    = c_ia* <psi|ia|walker> + 1/2 c_ia c_jb* <psi|ijab|walker>
    '''
    ci1 = ci1.conj()
    nocc = walker.shape[1]
    green_ov = _green(walker, slater)[:nocc, nocc:]
    cig = oe.contract('ia,ja->ij', ci1, green_ov, backend='jax')
    o0 = _walker_olp(walker, slater)
    o1 = oe.contract("ii->", cig, backend="jax")
    o2_1 = o1**2
    o2_2 = -oe.contract("ij,ji->", cig, cig, backend="jax")
    o2 = 2*o2_1 + o2_2
    return (1.0 + 2*o1 + o2) * o0

@jit
def _ci_exp_h1_dc(x, h1_mod: jax.Array, walker: jax.Array, slater: jax.Array, ci1: jax.Array) -> complex:
    '''
    <exp(T1)HF|(1+ci1+ci2) exp(x*h1_mod)|walker>
    '''
    t = x * h1_mod
    walker_1x = walker + t.dot(walker)
    o_exp = _ci_walker_olp_dc(walker_1x, slater, ci1)
    return o_exp 

@jit
def _ci_exp_h2_dc(x, chol_i: jax.Array, walker: jax.Array, slater: jax.Array, ci1: jax.Array) -> complex:
    '''
    <exp(T1)HF|(1+ci1+ci2) exp(x*h2)|walker>
    '''
    walker_2x = (
            walker
            + x * chol_i.dot(walker)
            + x**2 / 2.0 * chol_i.dot(chol_i.dot(walker))
        )
    o_exp = _ci_walker_olp_dc(walker_2x, slater, ci1)

    return o_exp

@jit
def _d2_ci_exp_h2i_dc(chol_i: jax.Array, walker: jax.Array, slater: jax.Array, ci1: jax.Array):
    x = 0.0
    f = lambda a: _ci_exp_h2_dc(a, chol_i, walker, slater, ci1)
    _, d2f = jax.jvp(lambda x: jax.jvp(f, [x], [1.0])[1], [x], [1.0])
    return d2f


@partial(jit, static_argnums=0)
def _calc_energy_ad_dc(trial, walker, ham_data, wave_data, ci1):

    norb = trial.norb
    h0 = ham_data['h0']
    h1_mod = ham_data['h1_mod']
    chol = ham_data["chol"].reshape(-1, norb, norb)
    trial_slater = wave_data['mo_t']

    # o0 = _walker_olp(walker, trial_slater)

    # one body
    f1 = lambda a: _ci_exp_h1_dc(a, h1_mod, walker, trial_slater, ci1)
    olp, d1_overlap = jvp(f1, [0.0], [1.0])

    # two body
    def scan_chol(carry, c):
        walker, trial_slater = carry
        return carry, _d2_ci_exp_h2i_dc(c, walker, trial_slater, ci1)

    _, d2_overlap_i = lax.scan(scan_chol, (walker, trial_slater), chol)
    d2_overlap = jnp.sum(d2_overlap_i)/2

    # <psi|(1+ci1+ci2) (h1+h2)|walker> / <psi|1+ci1+ci2|walker>
    e12 = (d1_overlap + d2_overlap) / olp

    return olp, h0 + e12

In [37]:
walker = jnp.array(np.random.rand(*prop_data['walkers'][0].shape))
slater = jnp.array(np.random.rand(*walker.shape))
ci1 = jnp.array(np.random.rand(*wave_data['t1'].shape))
ci2 = oe.contract('ia,jb->iajb', ci1, ci1, backend='jax')
olp1, e1 = _calc_energy_ad(trial, walker, ham_data, wave_data, ci1, ci2)
olp2, e2 = _calc_energy_ad_dc(trial, walker, ham_data, wave_data, ci1)
print(olp1, e1)
print(olp2, e2)

(0.01832378422286314+0j) (-95.97377481460448+0j)
(0.018323784222863144+0j) (-95.97377481460445+0j)


In [40]:
import timeit
%timeit _calc_energy_ad_dc(trial, walker, ham_data, wave_data, ci1)

12.7 ms ± 324 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [39]:
%timeit _calc_energy_ad(trial, walker, ham_data, wave_data, ci1, ci2)

11.4 ms ± 476 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [72]:
@partial(jit, static_argnums=0)
def _calc_energy_cisd(
    trial, 
    walker: jax.Array, 
    ham_data: dict, 
    wave_data: dict,
    ci1:  jax.Array,
    ci2:  jax.Array,
    ):

    '''
    A local energy evaluator for <psi(ci1+ci2)HF|H|walker> / <psi(ci1+ci2)|walker>
    all operators and the walkers and psi are in the same basis (normally MO)
    |psi> is not necesarily diagonal
    
    all green's function and the chol and ci coeff are as their original definition
    no half rotation performed
    '''

    nocc = trial.nelec[0]
    h0  = ham_data['h0']
    trial_slater = wave_data["mo_t"]
    h1 = (ham_data["h1"][0] + ham_data["h1"][1]) / 2.0
    chol = ham_data["chol"].reshape(-1, trial.norb, trial.norb)
    green = _green(walker, trial_slater) # full green
    green_ov = green[:nocc, nocc:]
    greenp = (green - jnp.eye(trial.norb))[:, nocc:]
    
    ci1 = ci1.conj() # applied to the bra
    ci2 = ci2.conj() # applied to the bra
    
    ##################### ref terms #########################
    # one-body 
    hg = oe.contract("pq,pq->", h1, green, backend="jax")
    e1_0 = 2 * hg

    # two-body
    gl = oe.contract("pr,gqr->gpq", green, chol, backend="jax")
    trgl = oe.contract('gpp->g', gl, backend="jax")
    e2_0_1 = 2 * jnp.sum(trgl**2)
    e2_0_2 = -oe.contract('gpq,gqp->',gl,gl, backend="jax")
    e2_0 = e2_0_1 + e2_0_2
    ##########################################################

    ######################### ci terms #######################
    # one-body single excitations 
    ci1g = oe.contract("ia,ia->", ci1, green_ov, backend="jax")
    e1_1_1 = 4 * ci1g * hg # c_ia G_ia G_pq h_pq
    gpci1 = oe.contract("pa,ia->pi", greenp, ci1, backend="jax")
    ci1_green = oe.contract("pi,iq->pq", gpci1, green[:nocc,:], backend="jax")
    e1_1_2 = -2 * oe.contract("pq,pq->", h1, ci1_green, backend="jax") # c_ia Gp_pa G_iq h_pq
    e1_1 = e1_1_1 + e1_1_2 # <psi|ci1 h1|walker> / <psi|walker>

    # one-body double excitations
    t2g_c = oe.contract("iajb,ia->jb", ci2, green_ov, backend="jax")
    t2g_e = oe.contract("iajb,ib->ja", ci2, green_ov, backend="jax")
    t2_green_c = (greenp @ t2g_c.T) @ green[:nocc,:]
    t2_green_e = (greenp @ t2g_e.T) @ green[:nocc,:]
    t2_green = 2 * t2_green_c - t2_green_e
    t2g = 2 * t2g_c - t2g_e
    gt2g = oe.contract("ia,ia->", t2g, green_ov, backend="jax")
    e1_2_1 = 2 * hg * gt2g
    e1_2_2 = -2 * oe.contract("ij,ij->", h1, t2_green, backend="jax")
    e1_2 = e1_2_1 + e1_2_2 # <exp(T1)HF|T2 h1|walker> / <exp(T1)HF|walker>

    # two-body single excitations
    e2_1_1 = 2 * e2_0 * ci1g # c_ia G_ia G_pr G_ps L_pr L_ps
    lci1g = oe.contract("gpq,pq->g", chol, ci1_green, backend="jax")
    e2_1_2 = -2 * oe.contract("g,g->",lci1g, trgl, backend="jax") # c_ia Gp_pa G_ir G_qs L_pr L_qs

    lci1 = oe.contract("gpa,ia->gpi", chol[:, :, nocc:], ci1, backend="jax")
    lg1 = oe.contract("gpr,qr->gpq", chol, green[:nocc,:], backend="jax")
    lci1g = oe.contract("gri,pr->gip", lci1, green, backend="jax")
    glgpci1 = jnp.einsum("gpr,ri->gpi", gl, gpci1, optimize="optimal")
    e2_1_3 = jnp.einsum("gpi,gpi->", glgpci1, lg1, optimize="optimal")
    e2_1 = e2_1_1 + 2 * (e2_1_2 + e2_1_3) # <exp(T1)HF|ci1 h2|walker> / <exp(T1)HF|walker>

    # two-body double excitations
    e2_2_1 = e2_0 * gt2g
    lt2g = oe.contract("gij,ij->g", chol, t2_green, backend="jax")
    e2_2_2_1 = -oe.contract("g,g->", lt2g, trgl, backend="jax")

    def scan_aux(carry, x):
        chol_i, gl_i = x
        lt2_green_i = oe.contract("pr,qr->pq",chol_i,t2_green,backend="jax")
        carry[0] += 0.5 * oe.contract("pq,pq->",gl_i,lt2_green_i,backend="jax")
        glgp_i = oe.contract("iq,qa->ia", gl_i[:nocc,:],greenp,backend="jax")
        l2t2_1 = oe.contract("ia,jb,iajb->",glgp_i,glgp_i,ci2,backend="jax")
        l2t2_2 = oe.contract("ib,ja,iajb->",glgp_i,glgp_i,ci2,backend="jax")
        carry[1] += 2 * l2t2_1 - l2t2_2
        return carry, 0.0

    [e2_2_2_2, e2_2_3], _ = lax.scan(scan_aux, [0.0, 0.0], (chol, gl))

    e2_2_2 = 4 * (e2_2_2_1 + e2_2_2_2)
    e2_2 = e2_2_1 + e2_2_2 + e2_2_3

    o0 = _walker_olp(walker, trial_slater)
    olp = (1.0 + 2*ci1g +  gt2g) * o0 # <(1+c1+c2)psi|walker>
    # <exp(T1)HF|h1+h2|walker> + <exp(T1)ci1 HF|(h1+h2)|walker> + <exp(T1)ci2 HF|(h1+h2)|walker>
    energy = h0 + (e1_0 + e2_0 + e1_1 + e2_1 + e1_2 + e2_2) / (1.0 + 2*ci1g +  gt2g)

    return olp, energy

In [71]:
@partial(jit, static_argnums=0)
def _calc_energy_cisd_dc(
    trial, 
    walker: jax.Array, 
    ham_data: dict, 
    wave_data: dict,
    ci1:  jax.Array,
    ):

    '''
    Disconnected Doubles!!! c_iajb = c_ia c_jb
    A local energy evaluator for <psi(ci1+ci2)HF|H|walker> / <psi(ci1+ci2)|walker>
    all operators and the walkers and psi are in the same basis (normally MO)
    |psi> is not necesarily diagonal
    
    all green's function and the chol and ci coeff are as their original definition
    no half rotation performed
    '''

    nocc = trial.nelec[0]
    h0  = ham_data['h0']
    # trial_slater = wave_data["mo_t"]
    h1 = (ham_data["h1"][0] + ham_data["h1"][1]) / 2.0
    chol = ham_data["chol"].reshape(-1, trial.norb, trial.norb)
    green = _green(walker, wave_data["mo_t"]) # full green
    green_ov = green[:nocc, nocc:]
    greenp = (green - jnp.eye(trial.norb))[:, nocc:]
    
    ci1 = ci1.conj() # applied to the bra
    # ci2 = ci2.conj() # applied to the bra
    
    ##################### ref terms #########################
    # one-body 
    gh = oe.contract("pr,qr->pq", h1, green, backend="jax")
    tr_gh = oe.contract("pp->", gh, backend="jax")
    e1_0 = 2 * tr_gh

    # two-body 
    gl = oe.contract("pr,gqr->gpq", green, chol, backend="jax")
    trgl = oe.contract('gpp->g', gl, backend="jax")
    e2_0_1 = 2 * jnp.sum(trgl**2)
    e2_0_2 = -oe.contract('gpq,gqp->',gl,gl, backend="jax")
    e2_0 = e2_0_1 + e2_0_2
    ##########################################################

    ######################### ci terms #######################
    # universal terms #
    cig = oe.contract("ia,ja->ij", ci1, green_ov, backend="jax")
    cigp = oe.contract("ia,pa->ip", ci1, greenp, backend="jax")
    
    o0 = _walker_olp(walker, wave_data["mo_t"])
    o1 = oe.contract("ii->", cig, backend="jax") # c_ia G_ia
    o2_1 = o1**2
    o2_2 = -oe.contract("ij,ji->", cig, cig, backend="jax") # c_ia G_ja c_jb G_ib
    o2 = 2*o2_1 + o2_2

    olp = (1.0 + 2*o1 + o2) * o0
    ###################

    # one-body single excitations 
    e1_1_1 = 4 * o1 * tr_gh # c_ia G_ia G_pq h_pq
    cigp_g = oe.contract("ip,iq->pq", cigp, green[:nocc,:], backend="jax") # c_ia Gp_pa G_ir
    e1_1_2 = -2 * oe.contract("pq,pq->", cigp_g, h1, backend="jax") # c_ia Gp_pa G_iq h_pq
    e1_1 = e1_1_1 + e1_1_2 # <psi|ci1 h1|walker> / <psi|walker>

    # one-body double excitations

    t2_green_c = o1 * oe.contract('jp,jq->pq', cigp, green[:nocc,:], backend='jax')
    t2_green_e = oe.contract('ji,ip,jq->pq', cig, cigp, green[:nocc,:], backend='jax')
    t2_green = 2 * t2_green_c - t2_green_e
    e1_2_1 = 2 * o2 * tr_gh
    e1_2_2 = -2 * oe.contract("ij,ij->", h1, t2_green, backend="jax")
    e1_2 = e1_2_1 + e1_2_2 # <exp(T1)HF|T2 h1|walker> / <exp(T1)HF|walker>

    # two-body single excitations
    e2_1_1 = 2 * o1 * e2_0 # c_ia G_ia G_pr G_ps L_pr L_ps

    lci1g = oe.contract("gpq,pq->g", chol, cigp_g, backend="jax") # c_ia Gp_pa G_ir L_pr
    e2_1_2 = -2 * oe.contract("g,g->", lci1g, trgl, backend="jax") # c_ia Gp_pa G_ir G_qs L_pr L_qs

    lci1 = oe.contract("gpa,ia->gpi", chol[:, :, nocc:], ci1, backend="jax")
    lg1 = oe.contract("gpr,qr->gpq", chol, green[:nocc,:], backend="jax")
    lci1g = oe.contract("gri,pr->gip", lci1, green, backend="jax")
    glgpci1 = jnp.einsum("gpr,ir->gpi", gl, cigp, optimize="optimal")
    e2_1_3 = jnp.einsum("gpi,gpi->", glgpci1, lg1, optimize="optimal")
    e2_1 = e2_1_1 + 2 * (e2_1_2 + e2_1_3) # <exp(T1)HF|ci1 h2|walker> / <exp(T1)HF|walker>

    # two-body double excitations
    e2_2_1 = o2 * e2_0
    lt2g = oe.contract("gij,ij->g", chol, t2_green, backend="jax")
    e2_2_2_1 = -oe.contract("g,g->", lt2g, trgl, backend="jax")

    def scan_aux(carry, x):
        chol_i, gl_i = x
        lt2_green_i = oe.contract("pr,qr->pq", chol_i, t2_green, backend="jax")
        carry[0] += 0.5 * oe.contract("pq,pq->", gl_i, lt2_green_i, backend="jax")
        glgp_i = oe.contract("iq,qa->ia", gl_i[:nocc,:], greenp, backend="jax")
        glgp_ci_i = oe.contract("ia,ja->ij", glgp_i, ci1, backend="jax")
        l2t2_1 = oe.contract("ii->", glgp_ci_i, backend="jax")**2
        l2t2_2 = oe.contract("ij,ji->", glgp_ci_i, glgp_ci_i, backend="jax")
        carry[1] += 2 * l2t2_1 - l2t2_2
        return carry, 0.0

    [e2_2_2_2, e2_2_3], _ = lax.scan(scan_aux, [0.0, 0.0], (chol, gl))

    e2_2_2 = 4 * (e2_2_2_1 + e2_2_2_2)
    e2_2 = e2_2_1 + e2_2_2 + e2_2_3

    energy = h0 + (e1_0 + e2_0 + e1_1 + e2_1 + e1_2 + e2_2) / (1.0 + 2*o1 + o2)

    return olp, energy

In [76]:
walker = jnp.array(np.random.rand(*walker.shape))
walker = 0.5j * (walker + jnp.array(np.random.rand(*walker.shape))*1j)

ci1 = jnp.array(np.random.rand(*wave_data['t1'].shape))
ci1 = 0.5 * (ci1 + jnp.array(np.random.rand(*wave_data['t1'].shape)*1j))
ci2 = oe.contract('ia,jb->iajb', ci1, ci1, backend='jax')

olp1, e1 = _calc_energy_ad(trial, walker, ham_data, wave_data, ci1, ci2)
olp2, e2 = _calc_energy_ad_dc(trial, walker, ham_data, wave_data, ci1)
print(olp1, e1)
print(olp2, e2)

(3.788573855709661e-05-3.2729951117825155e-05j) (-95.38324840023415-7.913569832757796j)
(3.7885738557096625e-05-3.2729951117825155e-05j) (-95.38324840023415-7.913569832757762j)


In [77]:
olp3, e3 = _calc_energy_cisd(trial, walker, ham_data, wave_data, ci1, ci2)
print(olp3, e3)
olp4, e4 = _calc_energy_cisd_dc(trial, walker, ham_data, wave_data, ci1)
print(olp4, e4)

(3.78857385570966e-05-3.2729951117825135e-05j) (-95.3832484002342-7.913569832757737j)
(3.7885738557096625e-05-3.2729951117825155e-05j) (-95.38324840023417-7.91356983275777j)


In [53]:
_ci_walker_olp_dc(walker, wave_data['mo_t'], ci1)

Array(-0.00244248+0.j, dtype=complex128)